In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import os

torch.set_num_threads(8)
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

IMAGE_SIZE = 128
BATCH_SIZE = 16

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ToTensor(),
])
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=os.path.join(BASE_DIR, "medical_data", "train"), transform=train_transforms)
test_dataset  = datasets.ImageFolder(root=os.path.join(BASE_DIR, "medical_data", "test"),  transform=test_transforms)

targets = np.array(train_dataset.targets)
class_counts = np.bincount(targets)
sample_weights = (1.0 / class_counts)[targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=0)

def add_medical_noise(image_tensor, noise_factor=0.4):
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)

print(f"✅ Data ready | Train: {len(train_dataset)} | Test: {len(test_dataset)}")

✅ Data ready | Train: 5216 | Test: 624


In [2]:
class DeepMedicalVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(DeepMedicalVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  32,  3, stride=2, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 64,  3, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(256*8*8, latent_dim)
        self.fc_logvar = nn.Linear(256*8*8, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 256*8*8)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.ConvTranspose2d(64,  32,  3, stride=2, padding=1, output_padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.ConvTranspose2d(32,  1,   3, stride=2, padding=1, output_padding=1), nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        h = self.encoder(x).view(-1, 256*8*8)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        h2 = F.relu(self.fc_decode(z)).view(-1, 256, 8, 8)
        return self.decoder(h2), mu, logvar


class MedicalExpertCNN(nn.Module):
    def __init__(self):
        super(MedicalExpertCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256*8*8, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Load pretrained weights
healer = DeepMedicalVAE()
expert = MedicalExpertCNN()
healer.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "medical_healer.pth"), weights_only=True))
expert.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "medical_expert.pth"), weights_only=True))
print("✅ Both pretrained models loaded")

✅ Both pretrained models loaded


In [3]:
# Joint optimizer for BOTH models
all_params = list(healer.parameters()) + list(expert.parameters())
optimizer = optim.Adam(all_params, lr=1e-4)  # lower LR for fine-tuning
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

# Weighted classification loss
n_normal, n_pneumonia = class_counts[0], class_counts[1]
cls_weights = torch.tensor([n_pneumonia / n_normal, 1.0], dtype=torch.float32)
classification_loss = nn.CrossEntropyLoss(weight=cls_weights)

EPOCHS = 15
LAMBDA = 5.0  # weight for classification loss relative to VAE loss

print(f"🚀 End-to-End Fine-Tuning for {EPOCHS} epochs...")
print(f"   λ (CNN weight): {LAMBDA}")
print(f"   Class weights: NORMAL={cls_weights[0]:.2f}, PNEUMONIA={cls_weights[1]:.2f}\n")

for epoch in range(EPOCHS):
    healer.train()
    expert.train()
    total_vae_loss, total_cls_loss, correct, total_imgs = 0, 0, 0, 0

    for batch_idx, (clean_images, labels) in enumerate(train_loader):
        noisy_images = add_medical_noise(clean_images, 0.4)

        optimizer.zero_grad()

        # Forward through BOTH models
        healed_images, mu, logvar = healer(noisy_images)
        predictions = expert(healed_images)

        # Joint loss
        BCE = F.binary_cross_entropy(healed_images, clean_images, reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        vae_loss = BCE + 0.5 * KLD
        cls_loss = classification_loss(predictions, labels)
        total_loss = vae_loss + LAMBDA * cls_loss

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, 1.0)
        optimizer.step()

        total_vae_loss += vae_loss.item()
        total_cls_loss += cls_loss.item()
        predicted = predictions.argmax(dim=1)
        total_imgs += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 50 == 0:
            acc = 100 * correct / total_imgs
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx:>3}/{len(train_loader)} | Acc: {acc:.1f}%")

    acc = 100 * correct / total_imgs
    avg_cls = total_cls_loss / len(train_loader)
    scheduler.step(avg_cls)
    lr = optimizer.param_groups[0]['lr']
    print(f"✅ Epoch {epoch+1}/{EPOCHS} | Acc: {acc:.1f}% | CNN Loss: {avg_cls:.3f} | LR: {lr:.1e}\n")

# Save fine-tuned models
torch.save(healer.state_dict(), os.path.join(BASE_DIR, "models", "medical_healer_finetuned.pth"))
torch.save(expert.state_dict(), os.path.join(BASE_DIR, "models", "medical_expert_finetuned.pth"))
print("💾 Saved fine-tuned models!")

🚀 End-to-End Fine-Tuning for 15 epochs...
   λ (CNN weight): 5.0
   Class weights: NORMAL=2.89, PNEUMONIA=1.00

  Epoch 1/15 | Batch   0/326 | Acc: 100.0%
  Epoch 1/15 | Batch  50/326 | Acc: 92.0%
  Epoch 1/15 | Batch 100/326 | Acc: 92.0%
  Epoch 1/15 | Batch 150/326 | Acc: 92.7%
  Epoch 1/15 | Batch 200/326 | Acc: 93.0%
  Epoch 1/15 | Batch 250/326 | Acc: 93.2%
  Epoch 1/15 | Batch 300/326 | Acc: 92.9%
✅ Epoch 1/15 | Acc: 93.0% | CNN Loss: 0.154 | LR: 1.0e-04

  Epoch 2/15 | Batch   0/326 | Acc: 93.8%
  Epoch 2/15 | Batch  50/326 | Acc: 93.1%
  Epoch 2/15 | Batch 100/326 | Acc: 93.6%
  Epoch 2/15 | Batch 150/326 | Acc: 93.4%
  Epoch 2/15 | Batch 200/326 | Acc: 93.3%
  Epoch 2/15 | Batch 250/326 | Acc: 93.7%
  Epoch 2/15 | Batch 300/326 | Acc: 93.6%
✅ Epoch 2/15 | Acc: 93.7% | CNN Loss: 0.120 | LR: 1.0e-04

  Epoch 3/15 | Batch   0/326 | Acc: 100.0%
  Epoch 3/15 | Batch  50/326 | Acc: 93.8%
  Epoch 3/15 | Batch 100/326 | Acc: 93.9%
  Epoch 3/15 | Batch 150/326 | Acc: 94.1%
  Epoch 3/15

In [4]:
# Final pipeline evaluation
healer.eval()
expert.eval()

noisy_correct, healed_correct, clean_correct, total = 0, 0, 0, 0

with torch.no_grad():
    for clean_imgs, labels in test_loader:
        noisy_imgs = add_medical_noise(clean_imgs, 0.5)
        healed_imgs, _, _ = healer(noisy_imgs)

        clean_preds  = expert(clean_imgs).argmax(dim=1)
        noisy_preds  = expert(noisy_imgs).argmax(dim=1)
        healed_preds = expert(healed_imgs).argmax(dim=1)

        total += labels.size(0)
        clean_correct  += (clean_preds  == labels).sum().item()
        noisy_correct  += (noisy_preds  == labels).sum().item()
        healed_correct += (healed_preds == labels).sum().item()

print("=" * 50)
print(f"📊 FINAL RESULTS ({total} test images)")
print("=" * 50)
print(f"  🟢 Clean  Accuracy : {100*clean_correct/total:.2f}%")
print(f"  🔴 Noisy  Accuracy : {100*noisy_correct/total:.2f}%")
print(f"  🟢 Healed Accuracy : {100*healed_correct/total:.2f}%")
print(f"  🚀 Recovery        : +{100*(healed_correct-noisy_correct)/total:.2f}%")
print("=" * 50)

# Per-class breakdown
classes = test_dataset.classes
all_labels, all_healed = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        noisy = add_medical_noise(imgs, 0.5)
        healed, _, _ = healer(noisy)
        preds = expert(healed).argmax(dim=1)
        all_labels.extend(labels.tolist())
        all_healed.extend(preds.tolist())

for idx, name in enumerate(classes):
    c = sum(p == l == idx for p, l in zip(all_healed, all_labels))
    t = sum(l == idx for l in all_labels)
    print(f"  {name}: {c}/{t} = {100*c/t:.1f}%")

📊 FINAL RESULTS (624 test images)
  🟢 Clean  Accuracy : 87.66%
  🔴 Noisy  Accuracy : 62.50%
  🟢 Healed Accuracy : 81.09%
  🚀 Recovery        : +18.59%
  NORMAL: 137/234 = 58.5%
  PNEUMONIA: 368/390 = 94.4%
